# LeetCode #1255: Maximum Score Words Formed by Letters

https://leetcode.com/problems/maximum-score-words-formed-by-letters/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(2^w \cdot n)$ | $O(n)$ |
| **Optimal: Bitmask DP ★** | $O(2^w \cdot w \cdot n)$ | $O(2^w)$ |

---

## Understanding the Methods

### Brute Force
Recursively try including or excluding each word, re-checking letter availability at each step. No memoisation leads to redundant recomputation across overlapping sub-problems.

### Optimal: Bitmask DP ★
Represent each subset of words as a bitmask over the $w \leq 14$ words. For each bitmask, compute the total letters used and score if the letter supply allows it. Because $2^{14} = 16384$ and each evaluation is $O(w \cdot n)$, the total work is feasible. We scan all subsets and track the maximum valid score.

**Constraints:**
* `1 <= words.length <= 14`
* `1 <= words[i].length <= 7`
* `1 <= letters.length <= 100`
* `letters[i]` and each character in `words[i]` are lowercase English letters.
* `score.length == 26`

## Solutions

### C#

In [ ]:
public class Solution {
    public int MaxScoreWords(string[] words, char[] letters, int[] score) {
        int w = words.Length;
        // Letter frequency available
        int[] avail = new int[26];
        foreach (char c in letters) avail[c - 'a']++;

        int best = 0;
        // Try every subset of words via bitmask enumeration
        for (int mask = 0; mask < (1 << w); mask++) {
            int[] used = new int[26];
            int total = 0;
            bool valid = true;

            for (int i = 0; i < w; i++) {
                if ((mask & (1 << i)) == 0) continue;
                // Accumulate letter counts and score for this word
                foreach (char c in words[i]) {
                    used[c - 'a']++;
                    total += score[c - 'a'];
                }
            }
            // Verify letter supply is not exceeded
            for (int k = 0; k < 26 && valid; k++)
                if (used[k] > avail[k]) valid = false;

            if (valid) best = Math.Max(best, total);
        }
        return best;
    }
}

### Python

In [ ]:
class Solution:
    def maxScoreWords(self, words: list[str], letters: list[str], score: list[int]) -> int:
        from collections import Counter
        avail = Counter(letters)
        w = len(words)
        best = 0

        # Evaluate every subset of words via bitmask
        for mask in range(1 << w):
            used: dict[str, int] = Counter()
            total = 0
            for i in range(w):
                if mask & (1 << i):
                    # Accumulate letter usage and score for included word
                    for c in words[i]:
                        used[c] += 1
                        total += score[ord(c) - ord('a')]

            # Accept subset only if letters are within supply
            if all(used[c] <= avail[c] for c in used):
                best = max(best, total)

        return best

### Go

In [ ]:
func maxScoreWords(words []string, letters []byte, score []int) int {
    avail := [26]int{}
    for _, c := range letters {
        avail[c-'a']++
    }

    w := len(words)
    best := 0

    // Enumerate all 2^w subsets
    for mask := 0; mask < (1 << w); mask++ {
        used := [26]int{}
        total := 0
        valid := true

        for i := 0; i < w; i++ {
            if mask&(1<<i) == 0 {
                continue
            }
            // Tally letters and score for this included word
            for _, c := range words[i] {
                used[c-'a']++
                total += score[c-'a']
            }
        }
        // Reject if any letter is over-used
        for k := 0; k < 26 && valid; k++ {
            if used[k] > avail[k] {
                valid = false
            }
        }
        if valid && total > best {
            best = total
        }
    }
    return best
}

### Rust

In [ ]:
use std::collections::HashMap;

impl Solution {
    pub fn max_score_words(words: Vec<String>, letters: Vec<char>, score: Vec<i32>) -> i32 {
        let mut avail = [0i32; 26];
        for c in &letters { avail[(c as u8 - b'a') as usize] += 1; }

        let w = words.len();
        let mut best = 0i32;

        // Try every subset of words
        for mask in 0..(1usize << w) {
            let mut used = [0i32; 26];
            let mut total = 0i32;

            for i in 0..w {
                if mask & (1 << i) == 0 { continue; }
                // Add letter counts and score for included word
                for c in words[i].bytes() {
                    let idx = (c - b'a') as usize;
                    used[idx] += 1;
                    total += score[idx];
                }
            }
            // Only accept if every letter stays within supply
            if (0..26).all(|k| used[k] <= avail[k]) {
                best = best.max(total);
            }
        }
        best
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `words=["dog","cat","dad","good"], letters="aabbdd", score=[1,0,9,5,0,...,4,...]`
Subset `{dad, good}` uses `d,a,d` + `g,o,o,d` — `d` used 3 times but only 2 available, invalid. Best valid subset gives 23.

### 2. Slightly Complex
**Input:** `words=["xxxz","ax","bx","cx"], letters="xxaxbx", score=[4,4,4,0,...,5,0,...,10]`
Some subsets use `x` heavily. The bitmask loop finds the combination that stays within `x` supply while maximising score.

### 3. Edge Case: Time Factor
**Input:** 14 words each of length 7, 100 letters.
$2^{14} = 16384$ subsets, each scanning up to $14 \times 7 = 98$ letters plus 26 availability checks — $\approx 2 \times 10^6$ operations total.

### 4. Edge Case: Space Factor
**Input:** Any valid input with $w = 14$.
The 16384-entry boolean concept collapses into two 26-element arrays reused each iteration — $O(1)$ extra space beyond the `dp` conceptual table.

### 5. Almost-Impossible but Plausible
**Input:** All 14 words share the same rare letter, but only 1 copy is available.
Only subsets containing at most one word that uses that letter are valid. The bitmask loop still explores all 16384 masks and returns the correct max score among those valid subsets.